# Fine-tune FinBERT for Financial Sentiment
Trains on twitter_financial_news_sentiment dataset.
Handles class imbalance with weighted loss.
Saves model to HuggingFace Hub.

In [ ]:
# Install dependencies
!pip install transformers datasets scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score
from torch.nn import CrossEntropyLoss

# Label mapping
# Dataset: 0=bearish, 1=bullish, 2=neutral
# FinBERT: positive, negative, neutral
# We keep 0=negative, 1=positive, 2=neutral throughout
ID2LABEL = {0: 'negative', 1: 'positive', 2: 'neutral'}
LABEL2ID = {'negative': 0, 'positive': 1, 'neutral': 2}
BASE_MODEL = 'ProsusAI/finbert'
print('Setup complete')

In [ ]:
# Load data - upload train_sentiment.csv and test_set.csv to Kaggle
train_df = pd.read_csv('/kaggle/input/hackathon-sentiment/train_sentiment.csv')
test_df  = pd.read_csv('/kaggle/input/hackathon-sentiment/test_set.csv')

print(f'Train: {len(train_df)} rows')
print(f'Test:  {len(test_df)} rows')
print(f'Label distribution:\n{train_df.label.value_counts()}')

In [ ]:
# Compute class weights to handle imbalance
# More weight on minority classes (bearish, bullish)
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f'Class weights: {class_weights}')
# Expected: higher weight for 0 (bearish) and 1 (bullish)

In [ ]:
# Tokenize
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128  # tweets are short, 128 is enough
    )

train_ds = Dataset.from_pandas(train_df[['text','label']])
test_ds  = Dataset.from_pandas(test_df[['text','label']])

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)
print('Tokenization done')

In [ ]:
# Custom Trainer with weighted loss for class imbalance
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds)
    }

print('Trainer class ready')

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)
print('Model loaded')

In [ ]:
# Training arguments
# 3 epochs, small LR to avoid catastrophic forgetting
args = TrainingArguments(
    output_dir='./finbert-finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=50,
    fp16=True,  # faster on GPU
    report_to='none'
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)
print('Trainer ready')

In [ ]:
# Train — expect ~20-30 min on Kaggle T4 GPU
trainer.train()

In [ ]:
# Evaluate on locked test set
results = trainer.evaluate()
print(f"Final Macro F1: {results['eval_macro_f1']:.4f}")
print(f"Final Accuracy: {results['eval_accuracy']:.4f}")

In [ ]:
# Push to HuggingFace Hub
# Replace with your HF username
HF_USERNAME = 'your-hf-username'
MODEL_NAME  = f'{HF_USERNAME}/finbert-financial-sentiment'

from huggingface_hub import login
login(token='your-hf-token-here')  # paste your HF token

trainer.model.push_to_hub(MODEL_NAME)
tokenizer.push_to_hub(MODEL_NAME)
print(f'Model pushed to: https://huggingface.co/{MODEL_NAME}')